# Ch8 SFT（Supervised Fine-Tuning）教案

**课程名称：** SFT 指令微调：让模型学会遵循指令

**预计总时长：** 约 100 分钟

**源文件：** `Ch8_SFT/Ch8_SFT.ipynb`（共 36 个 Cell，Cell 0-35）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 环境准备 + SFT 概览 | Cell 0-7 | 8 min |
| 8-20 min | 理论：SFT 本质 + ChatML 格式 + Loss Masking 公式 | Cell 2-4, 8-10 | 12 min |
| 20-35 min | 数据工程：数据加载 + SFTDataset + Mask 可视化 | Cell 11-17 | 15 min |
| 35-45 min | 评估方法：生成式 vs NLL ranking | Cell 19-21 | 10 min |
| 45-50 min | **休息 + 回顾** | -- | 5 min |
| 50-70 min | SFT 训练：训练循环 + Loss 可视化 | Cell 22-24 | 20 min |
| 70-85 min | SFT 前后对比：精度 + 混淆矩阵 + 模型保存 | Cell 25-28 | 15 min |
| 85-95 min | 总结 + 练习 + 讨论 | Cell 29-35 | 10 min |

---

## 课前准备

- [ ] 确认 Python 3.11+ 和 PyTorch 已安装（`import torch; print(torch.__version__)`）
- [ ] 确认 `transformers` 已安装（`from transformers import AutoTokenizer, AutoModelForCausalLM`）
- [ ] 确认模型 `uer/gpt2-chinese-cluecorpussmall` 可下载或已缓存（vocab_size = 21128）
- [ ] 确认 `data/` 目录下有 `gpt2_sft_train.jsonl`、`gpt2_sft_val.jsonl`、`gpt2_sft_test.jsonl`
- [ ] 如有 GPU，确认 CUDA 可用：`print(torch.cuda.is_available())`
- [ ] 预跑一遍全部 Cell，确认 SFT 训练约 53 秒内完成（GPU），CPU 约 20-60 分钟
- [ ] 准备白板或屏幕画板，用于手绘 Loss Masking 示意图

---

## 第一段：开场与环境准备（Cell 0-7）

📍 运行 Cell 0-1（Markdown 导读 + 学习路线）、Cell 6-7（环境准备 + 模型加载代码）

⏱ 时间分配：8 分钟

🎯 本段目标
- 建立学习动机：预训练模型为什么还需要 SFT？
- 确认环境就绪（PyTorch、Transformers、GPU/CPU）
- 加载基座模型 `uer/gpt2-chinese-cluecorpussmall`

🗣 讲课话术

> 大家好！上一章我们从零做了预训练，得到了一个「会说话」的基座模型。但有个问题——它只会续写文本，你问它「今天天气怎么样？」它不会回答，而是继续写一段关于天气的文章。
>
> 今天的主题就是 **SFT（Supervised Fine-Tuning）**——用有监督数据教模型「如何进行对话」。打个比方：预训练是让小孩读了几万本书，SFT 是教他如何与人礼貌对话。读过书不代表会聊天，对吧？
>
> 我们使用的基座模型是 `uer/gpt2-chinese-cluecorpussmall`——一个中文 GPT-2，词表大小 21,128。任务是电商客服诉求分类，有 6 个类别：延迟发货、退款申请、地址修改、物流异常、售后咨询、发票问题。
>
> 先运行环境准备。（运行 Cell 6-7）看到输出了吗？模型加载完毕，设备显示 cuda（或 cpu）。注意 Cell 7 里我们刻意抑制了 HuggingFace 的一些非关键警告，这是工程实践中常见的做法。

👀 输出要点
- Cell 6：导入成功，无报错
- Cell 7：模型加载成功，设备信息输出
- 基座模型：`uer/gpt2-chinese-cluecorpussmall`，vocab_size = 21128

❓ 预判问题
- **Q：SFT 和 Fine-tuning 有什么区别？**
  A：Fine-tuning 是泛指在预训练模型基础上继续训练。SFT 特指用「指令-回复」对进行有监督微调，是 RLHF 流程（SFT -> RM -> PPO/DPO）的第一步。
- **Q：为什么用 GPT-2 而不是更大的模型？**
  A：教学目的——GPT-2 足够小，能在普通 GPU 上几十秒内完成训练。原理与大模型完全一致。

➡️ 转场

> 环境就绪。现在让我们理解 SFT 的核心理论——它到底改变了什么？

---

## 第二段：理论——SFT 本质 + ChatML 格式 + Loss Masking（Cell 2-4, 8-10）

📍 浏览 Cell 2-4（SFT 理论 + 损失函数）、Cell 8（ChatML 格式说明）、运行 Cell 9（特殊 token 定义）、浏览 Cell 10（目标函数与 Mask 公式）

⏱ 时间分配：12 分钟（SFT 本质 4 分钟 + ChatML 4 分钟 + Loss Masking 4 分钟）

🎯 本段目标
- 理解 SFT = 从「能力」到「行为」的桥梁
- 掌握 ChatML 对话格式的设计原理
- 理解 Loss Masking 的数学公式和工程实现

🗣 讲课话术

> 先看 Cell 2 的对比表。基础模型的训练目标是「预测下一个 token」，输入是任意文本，输出也是文本续写。SFT 后呢？输入变成结构化对话，输出变成有帮助的回复。**同样的 next-token prediction，但训练数据的格式变了。**
>
> Cell 3 有一个深刻的洞察：预训练后的模型已经具备了强大的语言能力——它「知道」大量事实、理解语法、甚至能做推理。但它有一个致命问题：**它不知道该如何与人对话**。SFT 就是教它「行为模式」。
>
> 现在看 ChatML 格式。Cell 8 解释了为什么需要结构化的对话模板。基座模型只理解连续文本——它不知道哪些是用户说的，哪些是助手该回复的。ChatML 用特殊 token 来标记边界。
>
> 运行 Cell 9，定义特殊 token。（运行 Cell 9）这里我们定义了 `<|startoftext|>`、`<|endoftext|>` 等 token。这些特殊标记就像对话的「标点符号」——告诉模型「现在轮到你回答了」。
>
> 最关键的概念来了——**Loss Masking**。看 Cell 4 和 Cell 10 的公式：
>
> $$\mathcal{L}_{\text{SFT}} = - \sum_{t=1}^{|y|} \log P_\theta( y_t \mid x, y_{<t} )$$
>
> 注意，这里只对 assistant 部分的 token $y$ 计算 loss，忽略 system 和 user 的部分 $x$。为什么？因为我们不希望模型学会「复述用户的话」，而是学会「回答用户的话」。
>
> 工程实现上，对不计入 loss 的 token，把 `labels` 设为 `-100`——这是 PyTorch CrossEntropyLoss 的 `ignore_index` 默认值。简单粗暴但非常有效。

👀 输出要点
- Cell 2：基础模型 vs SFT 后模型的对比表
- Cell 9：特殊 token 定义（`<|startoftext|>`, `<|endoftext|>` 等）
- Cell 10：SFT 目标函数 $\mathcal{L}_{\text{SFT}}$ 和 mask $w_t \in \{0,1\}$

❓ 预判问题
- **Q：为什么不训练 prompt 部分？训练了会怎样？**
  A：如果也对 prompt 计算 loss，模型会花资源学习「复述指令」而非「回答指令」。实验表明只训 assistant 部分效果更好，尤其在数据量有限时。
- **Q：-100 这个值有什么特别的？**
  A：PyTorch 的 `nn.CrossEntropyLoss` 默认 `ignore_index=-100`。任何 label 为 -100 的位置都不计入 loss。这是框架约定，没有数学含义。
- **Q：ChatML 是唯一的格式吗？**
  A：不是。不同模型用不同格式——Alpaca 格式、Vicuna 格式、Llama-2 的 `[INST]` 格式等。关键是保持训练和推理时格式一致。

➡️ 转场

> 理论讲完了，下面动手！我们先加载数据，然后亲眼看看 Loss Masking 在实际数据上长什么样。

---

## 第三段：数据工程——数据加载 + SFTDataset + Mask 可视化（Cell 11-17）

📍 运行 Cell 12（标签定义 + 数据加载）、Cell 13（标签分布可视化）、Cell 15（SFTDataset 类定义）、Cell 16（Mask 可视化）、Cell 17（Mask 统计图）

⏱ 时间分配：15 分钟（数据加载 4 分钟 + Dataset 构造 5 分钟 + Mask 可视化 6 分钟）

🎯 本段目标
- 了解电商客服分类任务的数据格式和分布
- 掌握 SFTDataset 的构造逻辑：prompt + answer + eos
- 直观理解 Loss Masking 的效果：哪些 token 被训练、哪些被遮蔽

🗣 讲课话术

> 运行 Cell 12。（运行 Cell 12）这里定义了 6 个类别标签：**延迟发货、退款申请、地址修改、物流异常、售后咨询、发票问题**。然后从 JSONL 文件加载数据——训练集 2,400 条、验证集 300 条、测试集 300 条。
>
> 运行 Cell 13 看标签分布。（运行 Cell 13）6 个类别各 400 条，完美均衡。这是教学用的理想数据集——实际场景中类别通常是不均衡的。
>
> 重点来了——Cell 15 的 `SFTDataset` 类。它做了三件事：
> 1. 把 instruction 组装成 prompt：`{instruction}\n\n助手：`
> 2. 把 prompt + answer + eos 拼接成 `input_ids`
> 3. 构造 `labels`：prompt 部分全部设为 -100，只有 answer + eos 部分保留真实 token id
>
> 运行 Cell 16，看 Mask 可视化。（运行 Cell 16）大家看这个输出——灰色的 `[MASK]` 是不计入 loss 的 prompt 部分，绿色的 `[TRAIN]` 是计入 loss 的 assistant 回复部分。一目了然！
>
> 运行 Cell 17 看统计。（运行 Cell 17）关键数据：**74% 的 token 被 mask（不参与训练），只有 26% 参与训练**。平均每条样本的监督比例约 25.4%。这意味着大部分 token 是 prompt 上下文，模型只需要学习如何生成 assistant 的回复。
>
> 运行 Cell 18 创建 DataLoader。（运行 Cell 18）batch_size = 8（GPU），数据准备完毕。

👀 输出要点
- Cell 12：6 个标签，train=2400 / val=300 / test=300
- Cell 13：标签分布图（每类 400 条）
- Cell 16：Mask 可视化——灰色 [MASK] vs 绿色 [TRAIN]
- Cell 17：74% masked，26% train，平均监督比例 25.4%
- Cell 18：batch_size=8（GPU）/ 4（CPU）

❓ 预判问题
- **Q：为什么只有 26% 的 token 参与训练？这样效率是不是很低？**
  A：这正是 SFT 的特点。prompt 部分提供上下文但不贡献梯度。实际中 prompt 通常比 response 长（尤其多轮对话），监督比例可能更低。这也是为什么 SFT 数据需要精心设计——每条数据只有少量 token 在「干活」。
- **Q：MAX_LENGTH = 160 会不会截断数据？**
  A：Cell 15 的 SFTDataset 有防护——如果截断后监督 token 为 0（全部 -100），会丢掉该样本。对于我们的短文本分类任务，160 足够了。
- **Q：如果出现 None token id 怎么办？**
  A：SFTDataset 有第二层防护——如果 tokenizer 返回 None，该样本会被丢弃并打印警告。这是「不出错」的关键工程细节。

➡️ 转场

> 数据准备好了。在开始训练之前，我们先定义评估方法——否则训完了不知道好不好。

---

## 第四段：评估方法——生成式 vs NLL Ranking（Cell 19-21）

📍 浏览 Cell 19（评估方法说明）、运行 Cell 20（generate_answer 函数）、运行 Cell 21（eval_acc + 混淆矩阵工具）

⏱ 时间分配：10 分钟（生成式 3 分钟 + NLL ranking 4 分钟 + 基线测试 3 分钟）

🎯 本段目标
- 理解两种评估方式的原理和适用场景
- 理解 NLL ranking 为何比直接生成更稳定
- 测试基座模型的基线准确率

🗣 讲课话术

> Cell 19 提出了一个关键问题：怎么评估 SFT 的效果？这里有两条线。
>
> **第一条：生成式输出**（直观但不稳定）。让模型真的生成回答，然后人眼看。问题是小模型生成时容易出现格式错误、重复循环、跑题——前面一个 token 选错，后面全部跑偏。
>
> **第二条：NLL ranking**（稳定且可量化）。对每个候选标签，计算「如果回复是这个标签，模型觉得有多合理」的 NLL 分数。选 NLL 最小（最合理）的标签。
>
> 打个比方：生成式评估像让学生写作文——容易跑题。NLL ranking 像给学生选择题——只问「哪个答案最合理」，更稳定。
>
> NLL ranking 有三个关键优势：
> 1. **不依赖解码质量**——不需要模型「自己生成」，只需要「打分」
> 2. **充分利用上下文**——每一步都喂真实 token（teacher-forcing）
> 3. **数值连续可比较**——不像生成式输出需要人工判断
>
> 运行 Cell 20-21 定义评估函数。然后我们测一下基座模型——还没做 SFT 时准确率是多少？结果是 **39.17%**。6 个类别随机猜是 16.7%，所以基座模型有一定的语言理解能力，但远远不够。

👀 输出要点
- Cell 20：`generate_answer` 函数定义（生成式评估）
- Cell 21：`eval_acc` 函数定义（NLL ranking 评估）+ `plot_confusion` 工具
- 基座模型基线准确率：39.17%（通过 NLL ranking）

❓ 预判问题
- **Q：NLL ranking 只适用于分类任务吗？**
  A：是的，它需要有限的候选集。对于开放式生成（翻译、摘要），通常用 BLEU、ROUGE 等指标，或者人工评估。
- **Q：39.17% 的基线是怎么来的？**
  A：基座模型虽然没经过 SFT，但它在预训练时见过大量中文文本，对某些类别名称有一定的语义理解。比如看到「快递一直没到」可能会觉得「物流异常」比「发票问题」更合理。
- **Q：为什么不直接用 accuracy 评估生成式输出？**
  A：因为生成式输出的格式不确定——模型可能输出「类别：退款申请」也可能输出「我觉得这是退款」甚至乱码。解析不稳定会引入大量噪声。

➡️ 转场

> 基线 39.17%，记住这个数字。现在进入最核心的环节——SFT 训练！

---

## 休息 + 回顾（第 45-50 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. SFT 的本质是用「指令-回复」对教模型对话行为，核心技术是 **Loss Masking**——只对 assistant 回复部分计算 loss，用 labels=-100 屏蔽 prompt 部分。
2. 我们的数据集有 2,400 条训练样本（6 类各 400 条），74% 的 token 被 mask，仅 26% 参与训练，平均监督比例 25.4%。
3. 评估采用 NLL ranking（稳定可量化），基座模型基线准确率 39.17%。

**下一段预告：** 我们要开始 SFT 训练，看 loss 从 0.89 一路降到 0.03，准确率从 39% 飙升到 100%。

---

## 第五段：SFT 训练（Cell 22-24）

📍 浏览 Cell 22（训练说明 + 时间预估表）、运行 Cell 23（训练主循环）、运行 Cell 24（Loss 可视化）

⏱ 时间分配：20 分钟（训练配置 3 分钟 + 训练过程 10 分钟 + Loss 分析 7 分钟）

🎯 本段目标
- 理解 SFT 训练的超参数选择
- 观察完整训练过程中 loss 的变化
- 理解 NaN/Inf 防护和梯度裁剪的工程意义

🗣 讲课话术

> 先看 Cell 22 的训练配置和时间预估。（浏览 Cell 22）
>
> 关键超参数：
> - 学习率 **LR = 2e-05**（GPU）——注意这比预训练的 3e-4 小了一个数量级。为什么？因为 SFT 是在已有权重上微调，步子太大会把预训练学到的知识「踩坏」。
> - **2 个 epoch，最多 600 步**，batch_size = 8
> - 安全防护：NaN/Inf loss 跳过 step、梯度裁剪 `clip_grad_norm_(model.parameters(), 1.0)`
>
> 好，运行 Cell 23 开始训练！（运行 Cell 23，等待训练完成）
>
> 大家看训练日志——
> - **初始 Loss 约 0.8941**。比预训练的 ln(V) 小很多——因为基座模型已经有语言能力了，不是从随机开始。
> - Loss 快速下降，最终降到 **0.0256**。
> - 整个训练耗时 **53.3 秒**（GPU）。
>
> 运行 Cell 24 看 Loss 可视化。（运行 Cell 24）蓝线一路下降，从 0.89 到 0.03。没有明显的过拟合信号——验证 loss 也在下降。这说明 2,400 条数据对于这个小任务是足够的。
>
> 和上一章预训练对比一下：预训练 loss 从 5.9 降到 0.07（85 倍），但验证集严重过拟合。SFT loss 从 0.89 降到 0.03（35 倍），但效果更实用——因为我们有明确的评估指标（分类准确率）。

👀 输出要点
- Cell 22：训练时间预估表（GPU 3-8 分钟，CPU 20-60 分钟）
- Cell 23 训练日志：
  - LR = 2e-05, Epochs = 2, MAX_STEPS = 600, batch_size = 8
  - 初始 Loss: 0.8941
  - 最终 Loss: 0.0256
  - 训练耗时: 53.3s（GPU）
  - 总步数: 600
- Cell 24：Loss 曲线可视化（moving average 平滑）

❓ 预判问题
- **Q：为什么 SFT 的学习率比预训练小？**
  A：SFT 是在预训练好的权重上微调。学习率太大会破坏已有知识（catastrophic forgetting）。通常 SFT 学习率是预训练的 1/10 到 1/100。
- **Q：600 步够吗？**
  A：2400 条数据 / batch_size 8 = 300 步/epoch，2 epoch = 600 步。刚好过完两遍数据。对于这个简单任务已经足够，实际中可能需要 3-5 epoch。
- **Q：NaN/Inf loss 什么时候会出现？**
  A：当梯度爆炸、学习率过大、或数据中有异常样本时。跳过这些 step 而不是崩溃退出，是工程中的标准做法。

➡️ 转场

> Loss 降下来了，但最终效果如何？让我们看看 SFT 前后的精度对比——这是最激动人心的部分。

---

## 第六段：SFT 前后对比 + 模型保存（Cell 25-28）

📍 运行 Cell 26（精度对比 + 样例输出）、运行 Cell 27（混淆矩阵可视化）、运行 Cell 28（模型保存）

⏱ 时间分配：15 分钟（精度对比 5 分钟 + 混淆矩阵 5 分钟 + 模型保存 5 分钟）

🎯 本段目标
- 直观感受 SFT 前后准确率的巨大提升
- 通过混淆矩阵理解模型在各类别上的表现
- 掌握 HuggingFace 模型保存的工程实践

🗣 讲课话术

> 运行 Cell 26。（运行 Cell 26）
>
> 大家看结果——
> - **基座模型准确率：39.17%**
> - **SFT 后准确率：100.00%**
> - **提升幅度：+60.83 个百分点！**
>
> 从 39% 到 100%，只用了 2,400 条训练数据、600 步训练、53 秒时间。这就是 SFT 的威力——不是教模型新知识，而是教它如何使用已有的知识。
>
> 看几条样例对比。基座模型的生成输出通常格式混乱、答非所问。SFT 后的模型严格按照「类别：xxx\n理由：xxx」的格式回答，每个类别都分对了。
>
> 运行 Cell 27 看混淆矩阵。（运行 Cell 27）SFT 后的混淆矩阵是完美的对角线——每个类别都 100% 正确分类。对比基座模型的混淆矩阵，可以看到很多类别被混淆（比如「延迟发货」和「物流异常」）。
>
> 当然，100% 准确率在实际场景中不常见——我们的任务比较简单，6 个类别区分度大，数据干净。但即使在更复杂的任务上，SFT 带来的提升通常也是显著的。
>
> 运行 Cell 28 保存模型。（运行 Cell 28）使用 HuggingFace 的 `save_pretrained` 方法，同时保存模型权重和 tokenizer。这样下次加载只需要一行 `from_pretrained`。

👀 输出要点
- Cell 26：
  - Base accuracy: 39.17%
  - SFT accuracy: 100.00%
  - Delta: +60.83%
  - 样例输出对比（base 格式混乱 vs SFT 格式工整）
- Cell 27：
  - Base 混淆矩阵：多个类别互相混淆
  - SFT 混淆矩阵：完美对角线
  - 一图三栏对比可视化
- Cell 28：模型保存到 `models/ch8_sft_gpt2_teaching/`

❓ 预判问题
- **Q：100% 准确率是不是过拟合了？**
  A：测试集也是 100%，说明模型真正学会了任务而非记住训练数据。这个任务的类别区分度大，2,400 条数据足够学会区分。但如果类别增加到 50 个或数据更嘈杂，准确率会下降。
- **Q：SFT 只能做分类吗？**
  A：不，SFT 可以用于任何对话任务——问答、翻译、摘要、代码生成等。分类只是最容易量化评估的任务，所以教学中常用。
- **Q：save_pretrained 保存了什么？**
  A：模型权重（`pytorch_model.bin` 或 `model.safetensors`）、模型配置（`config.json`）、tokenizer 文件（`vocab.txt`、`tokenizer_config.json` 等）。加载时只需 `AutoModelForCausalLM.from_pretrained(path)`。

➡️ 转场

> 训练完了、效果验证了、模型也保存了。最后我们回顾一下全流程，然后留练习题。

---

## 第七段：总结 + 练习 + 讨论（Cell 29-35）

📍 浏览 Cell 29（总结 + 核心概念图谱）、浏览 Cell 30-31（练习：实现 Loss Masking）、浏览 Cell 33（Extra 扩展）、可选运行 Cell 35（Loss Mask 可视化 Extra）

⏱ 时间分配：10 分钟（总结 3 分钟 + 练习 7 分钟）

🎯 本段目标
- 回顾 SFT 全流程
- 学生动手实现简化版 Loss Masking
- 预告下一章 LoRA 与量化

🗣 讲课话术

> 翻看 Cell 29 的总结图谱：
> ```
> Base Model -> ChatML 格式 -> Loss Masking -> SFT 训练 -> Instruction-following Model
> ```
> 这就是完整的 SFT 闭环。四个关键组件：格式化对话数据、只训 assistant 部分、监督微调、评估验证。
>
> 现在看 Cell 30-31 的练习。这是一个简化版的 Loss Masking 实现——给你 `input_ids` 和 `assistant_start_idx`，要求生成 `labels`：
> - `assistant_start_idx` 之前的位置 -> `labels = -100`（不计算 loss）
> - `assistant_start_idx` 及之后的位置 -> `labels = input_ids[i]`（计算 loss）
>
> 这道题看起来简单，但它是 SFT 最核心的技术点。大家动手试试。
>
> Cell 33 的 Extra 列了几个扩展方向：扩充数据集、多轮训练、ROUGE 评估、超参数调优。感兴趣的同学课后可以尝试。
>
> 下一章 Ch9，我们要学 **LoRA 和量化**——当模型太大、显存不够时，如何用参数高效微调（PEFT）以小博大。

**提示节奏**
- 0-2 分钟：自己思考和编码
- 2 分钟：提示——关键是用列表推导或循环，根据 index 判断填 -100 还是 input_ids[i]
- 4 分钟：给出参考实现——`labels = [-100]*assistant_start_idx + input_ids[assistant_start_idx:]`

**常见错误**
- 忘记 `assistant_start_idx` 本身应该被训练（包含在 assistant 部分）
- 返回的 labels 长度和 input_ids 不一致
- 使用了 in-place 修改 input_ids（应该创建新列表）

**验证标准**
- `len(labels) == len(input_ids)`
- `labels[:assistant_start_idx]` 全是 -100
- `labels[assistant_start_idx:]` 和 `input_ids[assistant_start_idx:]` 完全一致

❓ 预判问题
- **Q：LoRA 和 SFT 是什么关系？**
  A：LoRA 是一种参数高效的微调方法——不更新全部参数，只训练低秩增量矩阵。SFT + LoRA = 用少量可训练参数完成指令微调，显存需求大幅降低。
- **Q：实际生产中 SFT 数据怎么获取？**
  A：常见方式——（1）人工标注（最贵最好）；（2）用强模型生成弱模型的训练数据（Self-Instruct / Alpaca 方式）；（3）从用户反馈中筛选高质量对话；（4）开源数据集（ShareGPT、OpenAssistant 等）。
- **Q：多轮对话的 SFT 怎么做 Loss Masking？**
  A：原理一样——所有 user turn 设为 -100，所有 assistant turn 保留 label。多轮时需要注意对话模板的正确拼接和 turn 边界的准确标记。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，环境准备 + 模型加载 | 0-7 |
| 8 | 理论：SFT 本质 + ChatML + Loss Masking | 2-4, 8-10 |
| 20 | 数据工程：数据加载 + SFTDataset + Mask 可视化 | 11-17 |
| 35 | 评估方法：生成式 vs NLL ranking | 19-21 |
| 45 | **休息 + 回顾** | -- |
| 50 | SFT 训练：训练循环 + Loss 可视化 | 22-24 |
| 70 | SFT 前后对比：精度 + 混淆矩阵 + 模型保存 | 25-28 |
| 85 | 总结 + 练习 + 讨论 | 29-35 |

---

## 附录 B：关键数据快速参考

### 核心公式

$$\mathcal{L}_{\text{SFT}} = - \sum_{t=1}^{|y|} \log P_\theta( y_t \mid x, y_{<t} )$$

$$\mathcal{L}(\theta) = -\sum_{t=1}^{T} w_t \log p_\theta(y_t \mid y_{<t}), \quad w_t \in \{0, 1\}$$

### 模型规格

| 指标 | 值 |
|:---|:---|
| 基座模型 | uer/gpt2-chinese-cluecorpussmall |
| vocab_size | 21,128 |
| 任务 | 电商客服诉求分类（6 类） |

### 数据规模速查

| 指标 | 值 |
|:---|:---|
| 训练集 | 2,400 条 |
| 验证集 | 300 条 |
| 测试集 | 300 条 |
| 类别数 | 6（延迟发货/退款申请/地址修改/物流异常/售后咨询/发票问题） |
| 每类样本数 | 400（训练集） |
| Masked 比例 | 74% |
| 训练比例 | 26% |
| 平均监督比例 | 25.4% |

### 训练关键数值

| 指标 | 值 |
|:---|:---|
| 学习率 | 2e-05 |
| Epochs | 2 |
| MAX_STEPS | 600 |
| batch_size | 8（GPU）/ 4（CPU） |
| 初始 Loss | 0.8941 |
| 最终 Loss | 0.0256 |
| 训练耗时 | 53.3s（GPU） |

### 效果对比

| 指标 | Base 模型 | SFT 后 | Delta |
|:---|:---|:---|:---|
| NLL ranking 准确率 | 39.17% | 100.00% | +60.83% |

---

## 附录 C：应急预案

### 场景 1：模型下载失败

**症状：** Cell 7 报错 `OSError: Can't load model uer/gpt2-chinese-cluecorpussmall`

**应对：**
1. 检查网络连接，确认能访问 huggingface.co
2. 如果在国内，设置镜像：`os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`
3. 如果已缓存，检查 `~/.cache/huggingface/` 下是否有对应模型
4. 备选：提前下载模型到本地目录，修改路径为本地路径

### 场景 2：Transformers 版本不兼容

**症状：** Cell 7 加载模型时大量 warning 或报错

**应对：**
1. 确认 transformers 版本 >= 4.30：`pip show transformers`
2. 升级：`pip install -U transformers`
3. Cell 7 已有 warning 抑制代码，正常 warning 不影响运行

### 场景 3：训练时间过长（CPU）

**症状：** CPU 上训练超过 30 分钟

**应对：**
1. 将 `MAX_STEPS` 从 600 减少到 100-200
2. Loss 下降趋势依然可见，只是最终精度可能稍低
3. 核心概念（Loss Masking、ChatML）不受影响
4. 可预先跑好结果，课上只展示 log 和可视化

### 场景 4：数据文件找不到

**症状：** Cell 12 报错找不到 `data/gpt2_sft_train.jsonl`

**应对：**
1. 确认 `data/` 目录在仓库根目录下
2. 检查路径解析逻辑是否正确（notebook 的工作目录可能不同）
3. 可手动设置数据路径

### 场景 5：CUDA 内存不足

**症状：** `RuntimeError: CUDA out of memory`

**应对：**
1. 减小 `BATCH_SIZE`（从 8 改为 4 或 2）
2. 减小 `MAX_LENGTH`（从 160 改为 128）
3. 切换到 CPU（GPT-2 模型小，CPU 能跑，只是慢）
4. 关闭其他占用 GPU 显存的程序

### 场景 6：NaN Loss

**症状：** 训练日志中出现 `NaN` 或 `Inf` loss

**应对：**
1. 代码已内置防护——NaN/Inf loss 时自动跳过该 step
2. 如果频繁出现，降低学习率（如 1e-05）
3. 检查数据是否有异常样本（空指令、超长文本等）
4. 确认 `clip_grad_norm_` 梯度裁剪正常工作